# День 2. Поиск, классификация и оценка NLP-моделей

**Практический блокнот на 4 пары (4 × 90 минут).** Примеры основаны на технических формулировках по теории вероятностей и математической статистике.

## Результаты обучения
Студент умеет:
- ранжировать тексты по косинусному сходству;
- объяснять различия между TF–IDF и плотными эмбеддингами;
- строить базовый классификатор технических текстов;
- оценивать качество классификации по нескольким метрикам;
- анализировать ошибки, утечку данных и ограничения малого корпуса.

**Зависимость:** `scikit-learn`. Для дополнительного блока с готовыми эмбеддингами понадобится `sentence-transformers` и загрузка модели, поэтому он необязателен.

## Пара 1 из 4. Поиск похожих задач и косинусное сходство

### 1.1. Корпус технических формулировок

Подготовим набор задач, размеченных по тематическим разделам. Метки нужны для последующей проверки результатов, но при поиске по запросу алгоритм сначала использует только тексты.

In [ ]:
texts = [
    "Найти вероятность противоположного события по заданной вероятности события.",
    "Вычислить вероятность объединения двух независимых событий.",
    "Определить вероятность не менее трех успехов в серии испытаний Бернулли.",
    "Найти математическое ожидание и дисперсию биномиальной случайной величины.",
    "Вычислить вероятность попадания нормальной случайной величины в интервал.",
    "Оценить выборочное среднее и исправленную дисперсию по наблюдениям.",
    "Построить доверительный интервал для математического ожидания.",
    "Проверить гипотезу о равенстве математического ожидания заданному значению.",
    "Сравнить две выборки и проверить гипотезу о равенстве средних.",
    "Вычислить медиану, квартили и межквартильный размах выборки.",
    "Найти коэффициент корреляции между двумя количественными признаками.",
    "Оценить коэффициенты линейной регрессии методом наименьших квадратов.",
    "Проверить согласие эмпирического распределения с теоретическим.",
    "Оценить вероятность события методом Монте-Карло.",
    "Определить распределение суммы независимых случайных величин."
]
labels = [
    "probability", "probability", "distributions", "distributions", "distributions",
    "descriptive_stats", "intervals", "hypothesis_tests", "hypothesis_tests",
    "descriptive_stats", "correlation_regression", "correlation_regression",
    "hypothesis_tests", "probability", "distributions"
]
print("Документов:", len(texts))
for text, label in zip(texts, labels):
    print(f"[{label:24s}] {text}")

### 1.2. TF–IDF и косинусное сходство

Косинусное сходство двух векторов (x,y) вычисляется как

\[
\operatorname{cos}(x,y)=\frac{x^\top y}{\|x\|_2\|y\|_2}.
\]

Для TF–IDF оно сравнивает направления векторов признаков. Это удобная базовая линия для поиска похожих формулировок, но синонимы и математически эквивалентные выражения могут иметь низкое сходство, если они используют разные слова.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b",
    ngram_range=(1, 2),
    sublinear_tf=True
)
X = vectorizer.fit_transform(texts)

def search(query, top_k=5):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), labels[i], texts[i]) for i in order]

for score, label, text in search("найти вероятность для нормального распределения"):
    print(f"{score:.3f} | {label:24s} | {text}")

### 1.3. Косинус на бумаге

Пусть (A=(1,1,0)), (B=(1,0,1)), (C=(2,2,0)).

1. Вычислите (cos(A,B)).
2. Вычислите (cos(A,C)).
3. Объясните, почему масштабирование ненулевого вектора положительным числом не меняет косинусное сходство.
4. Почему нулевой вектор создаёт проблему для этой формулы?

In [ ]:
A = np.array([1, 1, 0], dtype=float)
B = np.array([1, 0, 1], dtype=float)
C = np.array([2, 2, 0], dtype=float)

def cosine_manual(x, y):
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denom) if denom else float("nan")

print("cos(A, B) =", cosine_manual(A, B))
print("cos(A, C) =", cosine_manual(A, C))

### 1.4. Исследование поиска

Проверьте запросы:
- «проверить значимость корреляции»;
- «оценить разброс данных»;
- «вероятность не более двух успехов»;
- «интервал для среднего при неизвестной дисперсии».

Для каждого запроса выпишите топ-3 результата и ответьте: нашёл ли алгоритм правильную тему? Какие слова повлияли на ранжирование? Какие запросы неудачны и почему?

**Задание на пару:** измените `ngram_range` с `(1, 1)` на `(1, 2)` и сравните выдачу. Объясните эффект.

## Пара 2 из 4. От TF–IDF к семантическим эмбеддингам

### 2.1. Почему мешок слов ограничен?

TF–IDF основан на лексическом совпадении. Формулировки «оценить разброс выборки» и «вычислить дисперсию наблюдений» близки по смыслу, но могут не иметь общих ключевых слов. Плотные эмбеддинги отображают текст в вектор, обученный отражать некоторые семантические отношения.

Эмбеддинг не является гарантией понимания: качество зависит от обучающих данных, языка, предметной области и способа применения модели.

In [ ]:
semantic_pairs = [
    ("оценить разброс выборки", "вычислить дисперсию наблюдений"),
    ("проверить нулевую гипотезу", "провести статистическую проверку предположения"),
    ("найти вероятность события", "вычислить шанс наступления события"),
    ("построить доверительный интервал", "вычислить выборочную медиану")
]

for a, b in semantic_pairs:
    score = cosine_similarity(
        vectorizer.transform([a]),
        vectorizer.transform([b])
    )[0, 0]
    print(f"TF-IDF cosine={score:.3f} | {a!r} <> {b!r}")

### 2.2. Дополнительный эксперимент с готовой моделью (необязательно)

В этом блоке используется пакет `sentence-transformers`. При первом запуске он может загрузить модель из интернета; в аудитории без доступа к сети блок пропускается. Название модели и её лицензию нужно проверить до использования в проекте.

Установка: `python -m pip install sentence-transformers`.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    # При первом запуске потребуется загрузка модели.
    embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    sample_texts = [
        "Оценить разброс выборки",
        "Вычислить дисперсию наблюдений",
        "Построить доверительный интервал для среднего",
        "Вычислить выборочную медиану"
    ]
    E = embedding_model.encode(sample_texts, normalize_embeddings=True)
    E_sim = E @ E.T
    print(np.round(E_sim, 3))
except ImportError:
    print("Не установлен sentence-transformers; пропустите блок или установите пакет отдельно.")
except Exception as exc:
    print("Модель не удалось загрузить или запустить:", type(exc).__name__, str(exc)[:250])

### 2.3. Сравнение подходов

Сравните лексический поиск TF–IDF и семантический поиск по эмбеддингам на 5–10 парах текстов.

| Критерий | TF–IDF | Эмбеддинги |
|---|---|---|
| Основа | Частоты слов и n-грамм | Плотные векторы модели |
| Интерпретируемость признаков | Обычно выше | Обычно ниже |
| Синонимы | Часто слабое совпадение | Могут сопоставляться по смыслу |
| Требования | Небольшие вычислительные затраты | Нужна модель и больше ресурсов |
| Ошибки | Лексические совпадения без смысла | Семантические сбои и доменные ошибки |

Таблица описывает типичные свойства, а не гарантированный результат каждой модели.

### 2.4. Практические вопросы

1. Почему нельзя напрямую сравнивать векторы, полученные разными моделями?
2. Как построить тестовый набор для оценки поиска похожих задач?
3. Как оценивать качество ранжирования: достаточно ли смотреть только на первую найденную задачу?
4. Какие термины матстатистики могут иметь узкое специальное значение?

**Мини-задание:** составьте 10 пар «исходная формулировка — перефразировка» и вручную отметьте, должны ли они считаться семантически близкими.

## Пара 3 из 4. Классификация технических текстов

### 3.1. Постановка задачи

Автоматически определим раздел, к которому относится условие задачи: вероятность, распределения, описательная статистика, интервальные оценки, проверка гипотез или регрессия. Это задача многоклассовой классификации текста.

Небольшой учебный корпус нужен для демонстрации кода, но не для достоверного сравнения алгоритмов. При малом числе примеров качество сильно зависит от конкретного разбиения.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.33, random_state=7, stratify=labels
)

clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b",
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))

### 3.2. Разберём прогнозы и вероятности классов

Модель может возвращать класс с максимальной оценкой вероятности. Эти оценки не следует автоматически трактовать как хорошо откалиброванные вероятности. Сначала исследуйте прогнозы на нескольких новых формулировках.

In [ ]:
new_queries = [
    "Вычислить вероятность ровно двух успехов в десяти испытаниях",
    "Найти несмещенную оценку дисперсии",
    "Проверить статистическую гипотезу о равенстве средних",
    "Оценить связь между двумя признаками с помощью корреляции"
]

for query, predicted in zip(new_queries, clf.predict(new_queries)):
    print(f"{predicted:24s} | {query}")

if hasattr(clf.named_steps["model"], "predict_proba"):
    probabilities = clf.predict_proba(new_queries)
    classes = clf.named_steps["model"].classes_
    print("\nОценки вероятностей по классам:")
    for query, row in zip(new_queries, probabilities):
        best = np.argsort(row)[::-1][:3]
        print(query)
        print([(classes[i], round(float(row[i]), 3)) for i in best])

### 3.3. Сравним признаки и параметры

Попробуйте:
- только униграммы: `ngram_range=(1, 1)`;
- униграммы и биграммы: `ngram_range=(1, 2)`;
- логистическую регрессию без `class_weight="balanced"`;
- другой параметр `C` у логистической регрессии.

Не выбирайте модель только по одному случайному разбиению. Зафиксируйте настройки и сравните ошибки на одинаковых тестовых данных.

In [ ]:
# Пример альтернативной модели с униграммами
clf_unigrams = Pipeline([
    ("tfidf", TfidfVectorizer(
        token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b",
        ngram_range=(1, 1)
    )),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
clf_unigrams.fit(X_train, y_train)
pred_uni = clf_unigrams.predict(X_test)
print("Unigrams accuracy:", round(accuracy_score(y_test, pred_uni), 3))
print("Unigrams report:")
print(classification_report(y_test, pred_uni, zero_division=0))

### 3.4. Задания к паре 3

1. Какие слова или словосочетания могут указывать на раздел «проверка гипотез»?
2. Почему нельзя обучать модель на тестовой части, а затем сообщать качество на ней?
3. Как изменится эксперимент, если в одном классе будет 100 текстов, а в другом — 10?
4. Добавьте по 5 новых текстов на два класса и повторите обучение.
5. Найдите случаи, когда классификатор выбирает близкий, но неправильный раздел.

## Пара 4 из 4. Метрики, валидация и анализ ошибок

### 4.1. Почему accuracy недостаточно?

Accuracy — доля правильных ответов. При дисбалансе классов она может выглядеть высокой даже для модели, которая плохо распознаёт редкий класс. Precision отвечает на вопрос, какая доля предсказаний данного класса верна; recall — какую долю объектов этого класса удалось найти; F1 объединяет precision и recall через гармоническое среднее.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay
)

print("Accuracy:", accuracy_score(y_test, pred))
p, r, f1, support = precision_recall_fscore_support(
    y_test, pred, labels=clf.named_steps["model"].classes_, zero_division=0
)
for label, precision, recall, score, n in zip(
    clf.named_steps["model"].classes_, p, r, f1, support
):
    print(f"{label:24s} precision={precision:.2f} recall={recall:.2f} "
          f"F1={score:.2f} support={n}")

### 4.2. Матрица ошибок

Матрица ошибок помогает увидеть, какие классы модель путает между собой. Для учебного набора отдельные ячейки могут соответствовать всего одному примеру, поэтому интерпретируйте её осторожно.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    labels=clf.named_steps["model"].classes_,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
    colorbar=False
)
plt.title("Матрица ошибок классификатора")
plt.tight_layout()
plt.show()

### 4.3. Анализ конкретных ошибок

Для каждого неправильного ответа нужно понять причину: мало примеров, пересекающиеся термины, неоднозначная разметка, слишком короткий текст или потеря информации при предобработке. Исправление ошибки должно опираться на диагноз, а не на случайное изменение параметров.

In [ ]:
errors = []
for text, true_label, predicted_label in zip(X_test, y_test, pred):
    if true_label != predicted_label:
        errors.append((text, true_label, predicted_label))

if not errors:
    print("На этой тестовой выборке ошибок нет. Это не доказывает, что модель надёжна.")
else:
    for text, true_label, predicted_label in errors:
        print("Текст:", text)
        print("Истинный класс:", true_label)
        print("Предсказанный класс:", predicted_label)
        print("-" * 60)

### 4.4. Кросс-валидация

Кросс-валидация помогает оценить устойчивость качества к выбору разбиения. На очень маленьком корпусе даже кросс-валидация не заменяет сбор репрезентативных данных. Для каждого класса нужно достаточное число примеров, а дубликаты и почти одинаковые формулировки не должны попадать одновременно в обучение и тест.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

min_class_count = min(Counter(labels).values())
n_splits = min(5, min_class_count)

if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=11)
    cv_scores = cross_val_score(clf, texts, labels, cv=cv, scoring="f1_macro")
    print("F1 macro по фолдам:", np.round(cv_scores, 3))
    print("Среднее F1 macro:", round(float(cv_scores.mean()), 3))
    print("Стандартное отклонение:", round(float(cv_scores.std()), 3))
else:
    print("Для стратифицированной кросс-валидации нужно минимум два объекта в каждом классе.")

### Итоговый мини-проект

**Задача:** разработать прототип классификатора или поисковой системы для технических задач по терверу/матстатистике.

Требования:
1. Собрать не менее 30–50 формулировок и определить схему классов.
2. Удалить дубликаты и согласовать правила разметки.
3. Сравнить поиск TF–IDF и классификацию.
4. Использовать train/test split или кросс-валидацию без утечки данных.
5. Представить accuracy, macro-F1 и матрицу ошибок.
6. Разобрать минимум три ошибки.
7. Описать ограничения данных и план улучшения.

### Вопросы для обсуждения
- Что в вашей задаче важнее: precision, recall или macro-F1?
- Почему качество на учебном корпусе не гарантирует качества на новых формулировках?
- Как избежать попадания перефразировок одного условия одновременно в train и test?
- В каких случаях семантический поиск полезнее классификации?
- Как проверять корректность автоматической разметки и объяснимость результата?